In [7]:
# !pip install cdsapi
# !pip install eccodes
# !pip install cfgrib eccodes
# !conda install -c conda-forge eccodes cfgrib xarray -y

### INWERSJA - POZNAŃ 2023

In [1]:
import cdsapi

dataset = "reanalysis-era5-pressure-levels"
request = {
    "product_type": ["reanalysis"],
    "variable": ["temperature"],
    "year": ["2023"],
    "month": [
        "01", "02", "03",
        "04", "05", "06",
        "07", "08", "09",
        "10", "11", "12"
    ],
    "day": [
        "01", "02", "03",
        "04", "05", "06",
        "07", "08", "09",
        "10", "11", "12",
        "13", "14", "15",
        "16", "17", "18",
        "19", "20", "21",
        "22", "23", "24",
        "25", "26", "27",
        "28", "29", "30",
        "31"
    ],
    "time": [
        "00:00", "06:00", "12:00",
        "18:00"
    ],
    "pressure_level": ["925", "1000"],
    "data_format": "grib",
    "download_format": "unarchived",
    "area": [52.54000, 16.80000, 52.28000, 17.06000]
}

client = cdsapi.Client(
    url = "https://cds.climate.copernicus.eu/api",
    key = "f55a5040-c43c-40a7-957a-f078b44210f3"
)
client.retrieve(dataset, request).download()

2026-04-24 15:29:25,666 INFO Request ID is f9b0866b-8f48-4df6-ba98-7f369e526d9c
2026-04-24 15:29:25,797 INFO status has been updated to accepted
2026-04-24 15:29:41,016 INFO status has been updated to running
2026-04-24 15:29:48,756 INFO status has been updated to successful


2886a49bedd5282dfbe02818bd25edc3.grib:   0%|          | 0.00/314k [00:00<?, ?B/s]

'2886a49bedd5282dfbe02818bd25edc3.grib'

In [17]:
import eccodes
import xarray as xr
import cfgrib

# Odczyt pliku GRIB przez xarray
ds_i_pz_2023 = xr.open_dataset(
    '2886a49bedd5282dfbe02818bd25edc3.grib',
    engine='cfgrib'
)

print(ds_i_pz_2023)

<xarray.Dataset> Size: 35kB
Dimensions:        (time: 1460, isobaricInhPa: 2, latitude: 1, longitude: 1)
Coordinates:
  * time           (time) datetime64[ns] 12kB 2023-01-01 ... 2023-12-31T18:00:00
    valid_time     (time) datetime64[ns] 12kB ...
  * isobaricInhPa  (isobaricInhPa) float64 16B 1e+03 925.0
  * latitude       (latitude) float64 8B 52.5
  * longitude      (longitude) float64 8B 17.0
    number         int64 8B ...
    step           timedelta64[ns] 8B ...
Data variables:
    t              (time, isobaricInhPa, latitude, longitude) float32 12kB ...
Attributes:
    GRIB_edition:            1
    GRIB_centre:             ecmf
    GRIB_centreDescription:  European Centre for Medium-Range Weather Forecasts
    GRIB_subCentre:          0
    Conventions:             CF-1.7
    institution:             European Centre for Medium-Range Weather Forecasts
    history:                 2026-04-24T15:36 GRIB to CDM+CF via cfgrib-0.9.1...


In [18]:
import pandas as pd

df_i_pz_2023 = ds_i_pz_2023.to_dataframe().reset_index()


# Wyświetl wszystkie wiersze i kolumny
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)

print(df_i_pz_2023.shape)  # ile wierszy x kolumn
df_i_pz_2023.head()

(2920, 8)


,time,isobaricInhPa,latitude,longitude,number,step,valid_time,t
0,2023-01-01 00:00:00,1000.0,52.5,17.0,0,0 days,2023-01-01 00:00:00,288.558350
1,2023-01-01 00:00:00,925.0,52.5,17.0,0,0 days,2023-01-01 00:00:00,285.520508
2,2023-01-01 06:00:00,1000.0,52.5,17.0,0,0 days,2023-01-01 06:00:00,287.873779
3,2023-01-01 06:00:00,925.0,52.5,17.0,0,0 days,2023-01-01 06:00:00,285.555420
4,2023-01-01 12:00:00,1000.0,52.5,17.0,0,0 days,2023-01-01 12:00:00,288.744141


In [21]:
# Rozdzielam na dwa DataFrame 
df_925 = df_i_pz_2023[df_i_pz_2023['isobaricInhPa'] == 925][['time','t']].rename(columns = {'t': 't_925'})
df_1000 = df_i_pz_2023[df_i_pz_2023['isobaricInhPa'] == 1000][['time','t']].rename(columns = {'t': 't_1000'})

# Łaczymy oba df, w jeden do obliczenia różnicy
df_delta_pz_2023 = pd.merge(df_925, df_1000, how='inner', on='time')

# kolumna t_delta = podaje rożnice w stopniach 
df_delta_pz_2023['t_delta'] = df_delta_pz_2023['t_1000']-df_delta_pz_2023['t_925']
df_delta_pz_2023['inversion_0/1'] = [1 if e > 0 else 0 for e in df_delta_pz_2023['t_delta']]

# wyciagamy sama date , do łatiwejszegeo grupowania i wyciagniecia dziennej sreniej 

df_delta_pz_2023['data'] = df_delta_pz_2023['time'].dt.date
df_delta_pz_2023.tail()

,time,t_925,t_1000,t_delta,inversion_0/1,data
1455,2023-12-30 18:00:00,274.288818,278.460693,4.171875,1,2023-12-30
1456,2023-12-31 00:00:00,274.857178,276.722412,1.865234,1,2023-12-31
1457,2023-12-31 06:00:00,276.422119,275.470459,-0.951660,0,2023-12-31
1458,2023-12-31 12:00:00,276.890625,278.947266,2.056641,1,2023-12-31
1459,2023-12-31 18:00:00,277.028320,278.831055,1.802734,1,2023-12-31


In [25]:
# EKSPORT DO PLIKU CSV 

# df_delta_pz_2023.to_csv('/Users/marcinurbanski/Desktop/jmu_repo/1_projekt_air_polution_repo/0_ready_raw_data/inwersja_pz_2023.csv')

In [9]:
# import pandas as pd

# # Konwersja do DataFrame (temperatura w Kelvinach -> Celsjusze)
# df = ds['t'].to_dataframe().reset_index()
# df['temperature_C'] = df['t'] - 273.15

# print(df.head(20))
# print(f"\nZakres dat: {df['time'].min()} → {df['time'].max()}")
# print(f"Poziomy ciśnienia: {df['isobaricInhPa'].unique()}")

                  time  isobaricInhPa  number   step  latitude  longitude  \
0  2024-01-01 00:00:00         1000.0       0 0 days      53.5       14.5   
1  2024-01-01 00:00:00          925.0       0 0 days      53.5       14.5   
2  2024-01-01 06:00:00         1000.0       0 0 days      53.5       14.5   
3  2024-01-01 06:00:00          925.0       0 0 days      53.5       14.5   
4  2024-01-01 12:00:00         1000.0       0 0 days      53.5       14.5   
5  2024-01-01 12:00:00          925.0       0 0 days      53.5       14.5   
6  2024-01-01 18:00:00         1000.0       0 0 days      53.5       14.5   
7  2024-01-01 18:00:00          925.0       0 0 days      53.5       14.5   
8  2024-01-02 00:00:00         1000.0       0 0 days      53.5       14.5   
9  2024-01-02 00:00:00          925.0       0 0 days      53.5       14.5   
10 2024-01-02 06:00:00         1000.0       0 0 days      53.5       14.5   
11 2024-01-02 06:00:00          925.0       0 0 days      53.5       14.5   

<hr style="height:8px;border:none;background-color:black;">

## INWERSJA - ŁÓDŹ 2023

In [14]:
import cdsapi

dataset = "reanalysis-era5-pressure-levels"
request = {
    "product_type": ["reanalysis"],
    "variable": ["temperature"],
    "year": ["2023"],
    "month": [
        "01", "02", "03",
        "04", "05", "06",
        "07", "08", "09",
        "10", "11", "12"
    ],
    "day": [
        "01", "02", "03",
        "04", "05", "06",
        "07", "08", "09",
        "10", "11", "12",
        "13", "14", "15",
        "16", "17", "18",
        "19", "20", "21",
        "22", "23", "24",
        "25", "26", "27",
        "28", "29", "30",
        "31"
    ],
    "time": [
        "00:00", "06:00", "12:00",
        "18:00"
    ],
    "pressure_level": ["925", "1000"],
    "data_format": "grib",
    "download_format": "unarchived",
    "area": [51.91000, 19.33000, 51.64000, 19.59000]
}

client = cdsapi.Client(
    url = "https://cds.climate.copernicus.eu/api",
    key = "f55a5040-c43c-40a7-957a-f078b44210f3"
)
client.retrieve(dataset, request).download()

2026-04-24 11:13:26,954 INFO Request ID is 84b0f6ab-4e30-4626-935d-0718cc8f6e72
2026-04-24 11:13:27,096 INFO status has been updated to accepted
2026-04-24 11:21:49,721 INFO status has been updated to running
2026-04-24 11:25:50,627 INFO status has been updated to successful


1b28650d2dd1a0766896ed79e643b383.grib:   0%|          | 0.00/314k [00:00<?, ?B/s]

'1b28650d2dd1a0766896ed79e643b383.grib'

In [3]:
import eccodes
import xarray as xr
import cfgrib

# Odczyt pliku GRIB przez xarray
ds_i_ld_2023 = xr.open_dataset(
    '1b28650d2dd1a0766896ed79e643b383.grib',
    engine='cfgrib'
)

print(ds_i_ld_2023)

<xarray.Dataset> Size: 35kB
Dimensions:        (time: 1460, isobaricInhPa: 2, latitude: 1, longitude: 1)
Coordinates:
  * time           (time) datetime64[ns] 12kB 2023-01-01 ... 2023-12-31T18:00:00
    valid_time     (time) datetime64[ns] 12kB ...
  * isobaricInhPa  (isobaricInhPa) float64 16B 1e+03 925.0
  * latitude       (latitude) float64 8B 51.75
  * longitude      (longitude) float64 8B 19.5
    number         int64 8B ...
    step           timedelta64[ns] 8B ...
Data variables:
    t              (time, isobaricInhPa, latitude, longitude) float32 12kB ...
Attributes:
    GRIB_edition:            1
    GRIB_centre:             ecmf
    GRIB_centreDescription:  European Centre for Medium-Range Weather Forecasts
    GRIB_subCentre:          0
    Conventions:             CF-1.7
    institution:             European Centre for Medium-Range Weather Forecasts
    history:                 2026-04-24T15:30 GRIB to CDM+CF via cfgrib-0.9.1...


In [5]:
import pandas as pd

df_i_ld_2023 = ds_i_ld_2023.to_dataframe().reset_index()


# Wyświetl wszystkie wiersze i kolumny
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)

print(df_i_ld_2023.shape)  # ile wierszy x kolumn
df_i_ld_2023.head()

(2920, 8)


,time,isobaricInhPa,latitude,longitude,number,step,valid_time,t
0,2023-01-01 00:00:00,1000.0,51.75,19.5,0,0 days,2023-01-01 00:00:00,287.689209
1,2023-01-01 00:00:00,925.0,51.75,19.5,0,0 days,2023-01-01 00:00:00,286.381836
2,2023-01-01 06:00:00,1000.0,51.75,19.5,0,0 days,2023-01-01 06:00:00,287.905029
3,2023-01-01 06:00:00,925.0,51.75,19.5,0,0 days,2023-01-01 06:00:00,285.934326
4,2023-01-01 12:00:00,1000.0,51.75,19.5,0,0 days,2023-01-01 12:00:00,289.496094


In [9]:
# Rozdzielam na dwa DataFrame 
df_925 = df_i_ld_2023[df_i_ld_2023['isobaricInhPa'] == 925][['time','t']].rename(columns = {'t': 't_925'})
df_1000 = df_i_ld_2023[df_i_ld_2023['isobaricInhPa'] == 1000][['time','t']].rename(columns = {'t': 't_1000'})

# Łaczymy oba df, w jeden do obliczenia różnicy
df_delta_ld_2023 = pd.merge(df_925, df_1000, how='inner', on='time')

# kolumna t_delta = podaje rożnice w stopniach 
df_delta_ld_2023['t_delta'] = df_delta_ld_2023['t_1000']-df_delta_ld_2023['t_925']

# Tworzymy nowa kolumne z wartosciami 0/1 dla wskazania czy w danym pomiarze inwersja była czy nie.
df_delta_ld_2023['inversion_0/1'] = [1 if e > 0 else 0 for e in df_delta_ld_2023['t_delta']]

# wyciagamy sama date , do łatiwejszegeo grupowania i wyciagniecia dziennej sreniej 

df_delta_ld_2023['data'] = df_delta_ld_2023['time'].dt.date
df_delta_ld_2023.tail()

,time,t_925,t_1000,t_delta,inversion_0/1,data
1455,2023-12-30 18:00:00,274.669678,278.540771,3.871094,1,2023-12-30
1456,2023-12-31 00:00:00,274.476318,278.167725,3.691406,1,2023-12-31
1457,2023-12-31 06:00:00,274.984619,279.105225,4.120605,1,2023-12-31
1458,2023-12-31 12:00:00,276.476562,279.394531,2.917969,1,2023-12-31
1459,2023-12-31 18:00:00,278.311523,278.430664,0.119141,1,2023-12-31


In [11]:
# EKSPORT DO PLIKU CSV 

# df_delta_ld_2023.to_csv('/Users/marcinurbanski/Desktop/jmu_repo/1_projekt_air_polution_repo/0_ready_raw_data/inwersja_ld_2023.csv')